In [1]:
import os
from langchain.chat_models import init_chat_model

os.environ["GROQ_API_KEY"] = os.getenv("GROQ_API_KEY")

model = init_chat_model("groq:qwen/qwen3-32b")
response = model.invoke("Why do parrots talk?")
response

AIMessage(content='<think>\nOkay, so the user is asking why parrots talk. Hmm, let me start by recalling what I know about parrots. I remember that they\'re known for their ability to mimic human speech. But why exactly do they do that? I think it might be related to their social behavior. Parrots are social animals, so maybe they talk to communicate with each other or with humans.\n\nWait, I\'ve heard that parrots have a part of their brain similar to humans called the neostriatum. That might be involved in their ability to mimic sounds. Also, they have a syrinx, which is their vocal organ, allowing them to produce a variety of sounds. So maybe their vocal anatomy plays a role.\n\nBut why would they mimic humans specifically? Maybe it\'s a form of social bonding. In the wild, parrots might imitate each other\'s calls to stay in contact or to signal safety. When they\'re kept as pets, they might imitate human speech to interact with their owners. It could be a way to get attention or r

In [2]:
from langchain.tools import tool


@tool
def get_weather(location:str)->str:
    """Get the weather at a loaction"""
    return f"its rainy in {location}"


model_with_tools=model.bind_tools([get_weather])


In [4]:
response = model_with_tools.invoke("What is the weather like in New York?")
print(response)
print("---"*50)
for tool_call in response.tool_calls:
    print(f"Tool Call: {tool_call['name']}")
    print(f"Args: {tool_call['args']}")


content='' additional_kwargs={'reasoning_content': 'Okay, the user is asking about the weather in New York. Let me check the tools available. There\'s a function called get_weather that takes a location parameter. Since the user specified New York, I need to call that function with "New York" as the location. I\'ll make sure the arguments are correctly formatted in JSON. No other tools are available, so this should be the only function call needed.\n', 'tool_calls': [{'id': '2smnjbg2g', 'function': {'arguments': '{"location":"New York"}', 'name': 'get_weather'}, 'type': 'function'}]} response_metadata={'token_usage': {'completion_tokens': 108, 'prompt_tokens': 156, 'total_tokens': 264, 'completion_time': 0.155501532, 'completion_tokens_details': {'reasoning_tokens': 83}, 'prompt_time': 0.012182404, 'prompt_tokens_details': None, 'queue_time': 0.046844396, 'total_time': 0.167683936}, 'model_name': 'qwen/qwen3-32b', 'system_fingerprint': 'fp_5cf921caa2', 'service_tier': 'on_demand', 'fin

# Tool Execution Loops

In [6]:
messages = [{"role": "user", "content": "What is the weather like in New York?"}]
ai_msg = model_with_tools.invoke(messages)
messages.append(ai_msg)

for tool_call in ai_msg.tool_calls:
    tool_result = get_weather.invoke(tool_call)
    messages.append(tool_result)

final_response = model_with_tools.invoke(messages)
print(final_response.text)

    


It's currently rainy in New York.


In [9]:
messages

[{'role': 'user', 'content': 'What is the weather like in New York?'},
 AIMessage(content='', additional_kwargs={'reasoning_content': 'Okay, the user is asking about the weather in New York. Let me check the tools available. There\'s a function called get_weather that takes a location parameter. Since the user specified New York, I need to call that function with "New York" as the location. I\'ll make sure to format the tool call correctly within the XML tags.\n', 'tool_calls': [{'id': 't7txx4gvv', 'function': {'arguments': '{"location":"New York"}', 'name': 'get_weather'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 95, 'prompt_tokens': 156, 'total_tokens': 251, 'completion_time': 0.129811473, 'completion_tokens_details': {'reasoning_tokens': 70}, 'prompt_time': 0.007047679, 'prompt_tokens_details': None, 'queue_time': 0.04657821, 'total_time': 0.136859152}, 'model_name': 'qwen/qwen3-32b', 'system_fingerprint': 'fp_5cf921caa2', 'service_tier': 'on_de